# GaMMA association of QC'd ELEP DAS picks (2-9 Hz, 3-month run)

2026-11-17. Copy of `gamma_ELEP_das_rainier_11092026_savethis copy.ipynb` with a pick QC added before the association
(the PyOcto counterpart, with the same QC, is `picking_rainier_das_pyocto_events_octo_ELEP_QC_11172026.ipynb`).

Pick QC (function `qc_elep_file`, tested in `temp_test_elep_pick_cleaning.ipynb`), per 1-minute continuous record and on
all 1107 ELEP-picked channels, before the cut to the station channels:

1. **Boundary**: drop a pick less than `edge_s = 1` s from the start or the end of its record.
2. **Coherence**: the record needs a run of at least `min_run = 7` consecutive ELEP-picked channels with a P pick, or
   with an S pick, in which neighbouring picks are at most `max_dt = 0.5` s apart. Otherwise all picks of the record
   are dropped; if it has one, all its remaining picks are kept.

Otherwise the same GaMMA setup as `gamma_phasenet_das_rainier_events946_10292026_savethis.ipynb` (the
946-event PhaseNet-DAS run), with the picks swapped for the ELEP run
`picking-denoising/3-month-run_2-9hz_10302026` (`ML/Phasepicking_MORA.py`: 2-9 Hz bandpass, ensemble of
six EQTransformer models combined by semblance, run finished 2026-11-06).

What changes with ELEP picks:

| item | ELEP (this notebook) | PhaseNet-DAS (946-event run) |
| --- | --- | --- |
| pick files | one CSV per hour, `MORADAS_YYYYMMDD_HH.csv` | one CSV per minute, `decimator_..._UTC.csv` |
| picked channels | every 4th channel (2, 6, 10, ...) | every channel |
| picks per channel | at most one P and one S per 1-minute record | any number |
| pick score | not saved (thresholds P > 0.10, S > 0.05 in the picker); `prob` set to 1 | `phase_score` |
| stations used | nearest ELEP channel to each PhaseNet station (c, or c + 2 when c % 4 == 0; at most 19 m away) | every 10th located channel: 320 |

Everything else is identical: the same 320-station layout, `min_picks_per_eq = 100`, `min_p_picks_per_eq = min_s_picks_per_eq = 0`,
BGMM with oversample 5, `dbscan_eps = 10 s`, IASP91 eikonal at h = 1 km, box lon -123 to -120,
lat 45 to 48, depth 0-40 km.

Outputs go to `.../gamma_association_11172026/rainier_mora_das_elep_qc/` (with a `_test<hours>h_<day>` suffix when
`test_window` is set), including `pick_qc_summary.csv` with the QC counts per hourly file.
**Re-running the association cell overwrites `gamma_events.csv` and `gamma_picks.csv` in that folder.**


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pyproj import Proj
from gamma.utils import association, estimate_eps

## 1. Data

In [2]:
## Paths: where the inputs live and where results and figures go
region = "rainier_mora_das_elep_qc"
## Quick test: (start day, hours). None runs the full 3-month window. 2023-08-27 is the busiest day of the
## deployment (23 PNSN events, incl. the M2.9 at 10:10:23 UTC); the first day, 08-23, has picks only from 19:19 UTC.
test_window = ("2023-08-27", 24)

run_dir = ("/1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/picking-denoising/"
           "3-month-run_2-9hz_10302026")
picks_dir = run_dir                                                   # MORADAS_YYYYMMDD_HH.csv
data_path = lambda x: os.path.join("data", x)
station_csv = data_path("Mora_channellocation.csv")
iasp91_csv = data_path("vmodel/IASP91_1.csv")
pnsn_csv = data_path("pnsn_catalog_aug_nov.csv")
comcat_csv = data_path("comcat_rainier_25km.csv")

result_root = ("/1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/"
               f"gamma_association_11172026/{region}"
               + (f"_test{test_window[1]}h_{test_window[0]}" if test_window else ""))
os.makedirs(result_root, exist_ok=True)
result_path = lambda x: os.path.join(result_root, x)
os.makedirs("figures", exist_ok=True)
figure_dir = lambda x: os.path.join("figures", x)

for name, p in [("picks", picks_dir), ("stations", station_csv), ("iasp91", iasp91_csv),
                ("pnsn", pnsn_csv), ("comcat", comcat_csv), ("results", result_root)]:
    print(f"{'ok' if os.path.exists(p) else 'MISSING':8s}{name:10s}{p}")

## Inventory of the pick files. ELEP writes one CSV per hour; the file name carries the hour, so a
## time window can be selected without opening the files.
files = sorted(f for f in os.listdir(picks_dir) if f.startswith("MORADAS_") and f.endswith(".csv"))
pick_files = pd.DataFrame({
    "file": files,
    "t_start": pd.to_datetime([f[len("MORADAS_"):-len(".csv")] for f in files], format="%Y%m%d_%H", utc=True),
}).sort_values("t_start").set_index("t_start")

print(f"\n{len(pick_files)} hourly pick files, {pick_files.index.min()} to {pick_files.index.max()}")
gap = pick_files.index.to_series().diff()
gaps = gap[gap > pd.Timedelta(hours=1)]
print(f"{len(gaps)} missing hours" + (f"; largest gap {gaps.max()} ending {gaps.idxmax()}" if len(gaps) else ""))
skipped = os.path.join(run_dir, "skipped_files.csv")
if os.path.exists(skipped):
    print(f"{sum(1 for _ in open(skipped))} unreadable raw files skipped by the picker (see skipped_files.csv)")


ok      picks     /1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/picking-denoising/3-month-run_2-9hz_10302026
ok      stations  data/Mora_channellocation.csv
ok      iasp91    data/vmodel/IASP91_1.csv
ok      pnsn      data/pnsn_catalog_aug_nov.csv
ok      comcat    data/comcat_rainier_25km.csv
ok      results   /1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/gamma_association_11172026/rainier_mora_das_elep_qc_test24h_2023-08-27

2184 hourly pick files, 2023-08-23 00:00:00+00:00 to 2023-11-21 23:00:00+00:00
0 missing hours
5 unreadable raw files skipped by the picker (see skipped_files.csv)


In [3]:
### Pick QC, applied to every hourly ELEP file before the picks are used
# Each row of an ELEP file is one channel in one 1-minute continuous record (trace_start_time).
# The QC works per 1-minute record, on all 1107 ELEP-picked channels (2, 6, 10, ...), i.e. before
# the cut to the station channels:
#   1. boundary: a pick closer than edge_s to the start or the end of its record is dropped.
#   2. coherence: with the boundary picks removed, the record needs a run of at least min_run
#      consecutive ELEP-picked channels with a P pick, or with an S pick. Two neighbouring channels
#      only count as the same run when their picks are at most max_dt seconds apart (slope
#      condition: an earthquake arrival is flat across neighbouring channels, a vehicle is not).
#      A record without such a run loses all its picks; a record with one keeps all its
#      remaining P and S picks.
edge_s = 1.0      # [s]
min_run = 7       # consecutive ELEP-picked channels (7 channels ~ 270 m of cable)
max_dt = 0.5      # [s] largest pick-time difference between neighbouring channels in a run
record_s = 60.0   # [s] length of one continuous record


def qc_elep_file(path, edge_s=1.0, min_run=7, record_s=60.0, max_dt=0.5):
    """Read one hourly ELEP file and return the picks that pass the QC, with the counts.

    Returns (picks, counts). picks has one row per kept pick, on all ELEP channels:
    station (channel number), phase ('P' / 'S'), time (UTC timestamp).
    """
    df = pd.read_csv(path, usecols=['station_code', 'trace_start_time', 'trace_p_arrival', 'trace_s_arrival'], dtype=str)
    df['station'] = df['station_code'].str[len('das'):].astype(int)
    df['record'] = pd.to_datetime(df['trace_start_time'], utc=True)
    # position of the channel in the list of ELEP-picked channels (2, 6, 10, ... -> 0, 1, 2, ...)
    df['ich'] = np.searchsorted(np.sort(df['station'].unique()), df['station'])

    picks = df.melt(id_vars=['record', 'station', 'ich'], value_vars=['trace_p_arrival', 'trace_s_arrival'],
                    var_name='phase', value_name='time').dropna(subset=['time'])
    picks['phase'] = picks['phase'].map({'trace_p_arrival': 'P', 'trace_s_arrival': 'S'})
    picks['time'] = pd.to_datetime(picks['time'], utc=True)
    offset = (picks['time'] - picks['record']).dt.total_seconds()

    # 1. boundary check
    edge = (offset < edge_s) | (offset > record_s - edge_s)

    # 2. coherence check: longest flat run of consecutive channels per record, in P or in S
    good = picks[~edge].assign(offset=offset[~edge]).sort_values(['record', 'phase', 'ich'])
    new_run = ((good['ich'].diff() != 1) | (good['record'] != good['record'].shift())
               | (good['phase'] != good['phase'].shift()) | (good['offset'].diff().abs() > max_dt))
    run_len = good.groupby(new_run.cumsum())['ich'].transform('size')
    ok_records = good.loc[run_len >= min_run, 'record'].unique()
    keep = ~edge & picks['record'].isin(ok_records)

    counts = {'records': df['record'].nunique(), 'records_kept': len(ok_records), 'picks': len(picks),
              'picks_boundary': int(edge.sum()), 'picks_incoherent_record': int((~edge & ~keep).sum()),
              'picks_kept': int(keep.sum())}
    return picks.loc[keep, ['station', 'phase', 'time']], counts

In [4]:
## Time window, from test_window in the paths cell.
import time
if test_window:
    t_start = pd.Timestamp(test_window[0], tz="UTC")
    t_end = t_start + pd.Timedelta(hours=test_window[1])
else:
    t_start = pick_files.index.min()
    t_end = pick_files.index.max() + pd.Timedelta(hours=1)    # full run
window_files = pick_files[(pick_files.index >= t_start) & (pick_files.index < t_end)]
print(f"window {t_start} .. {t_end}: {len(window_files)} hourly files")

## read stations: same station layout as the PhaseNet-DAS run (every 10th located channel, 320 stations).
## ELEP only picked every 4th channel (2, 6, 10, ...; ch_itv = 4 in ML/Phasepicking_MORA.py), so each
## PhaseNet station channel c is replaced by the nearest ELEP channel: c itself when c % 4 == 2
## (10, 30, 50, ...), otherwise c + 2 (0 -> 2, 20 -> 22, ...), i.e. at most 2 channels (19 m) away.
## The station position is that of the ELEP channel.
channel_stride = 10
geom = pd.read_csv(station_csv)
located = geom.dropna(subset=["Longitude [°]", "Latitude [°]"]).astype({"Channel": int}).set_index("Channel")
pn_channels = located.index[located.index % channel_stride == 0]
elep_channels = np.where(pn_channels % 4 == 2, pn_channels, pn_channels + 2)
keep = np.isin(elep_channels, located.index)
print(f"{len(pn_channels)} PhaseNet-DAS station channels; {(~keep).sum()} have no located ELEP neighbour and are dropped")
stations = pd.DataFrame({
    "station_id": elep_channels[keep],
    "phasenet_channel": pn_channels[keep],
    "longitude": located.loc[elep_channels[keep], "Longitude [°]"].values,
    "latitude": located.loc[elep_channels[keep], "Latitude [°]"].values,
    "elevation_m": located.loc[elep_channels[keep], "Elevation [m]"].values,
})
stations.to_csv(result_path("stations.csv"), index=False)
stations = stations.drop(columns="phasenet_channel").rename(columns={"station_id": "id"})
stations["id"] = stations["id"].astype(str)
print(f"{len(stations)} DAS stations out of {len(located)} located channels")
print("Station format:", stations.iloc[:5], sep="\n")

## read picks: one ELEP CSV per hour, one row per channel per 1-minute record, with station_code
## "das<channel>" and at most one P time (trace_p_arrival) and one S time (trace_s_arrival). Each file goes
## through qc_elep_file on all its channels, and the picks that pass are then cut to the station channels.
## ELEP does not save its pick probability (it only kept P > 0.10 and S > 0.05), so phase_score is 1.
t0 = time.time()
station_channels = set(stations["id"].astype(int))
chunks, qc_rows = [], []
for f in window_files["file"]:
    df, counts = qc_elep_file(os.path.join(picks_dir, f), edge_s, min_run, record_s, max_dt)   # picks that pass, all ELEP channels
    df = df[df["station"].isin(station_channels)]
    counts["picks_kept_on_stations"] = len(df)
    qc_rows.append(counts)
    if len(df):
        chunks.append(df.rename(columns={"station": "station_id", "phase": "phase_type", "time": "phase_time"}))
picks = pd.concat(chunks, ignore_index=True)
del chunks
print(f"read and QC'd {len(window_files)} pick files in {time.time() - t0:.0f} s")

qc_summary = pd.DataFrame(qc_rows, index=pd.Index(window_files["file"], name="file"))
qc_summary.to_csv(result_path("pick_qc_summary.csv"))
n = qc_summary.sum()
print(f"pick QC (edge_s={edge_s:g} s, min_run={min_run}, max_dt={max_dt:g} s), all ELEP channels:")
print(f"  {n['records_kept']} of {n['records']} 1-minute records kept ({100 * n['records_kept'] / n['records']:.1f}%)")
print(f"  {n['picks']} picks: {n['picks_boundary']} dropped at a record boundary, "
      f"{n['picks_incoherent_record']} dropped in records without a coherent run, "
      f"{n['picks_kept']} kept ({100 * n['picks_kept'] / n['picks']:.1f}%)")
print(f"  {n['picks_kept_on_stations']} kept picks on the {len(station_channels)} station channels")
picks["phase_score"] = 1.0
picks = picks.sort_values("phase_time").reset_index(drop=True)
picks.to_csv(result_path("picks.csv"), index=False, date_format="%Y-%m-%dT%H:%M:%S.%f")
picks.rename(columns={"station_id": "id", "phase_time": "timestamp", "phase_type": "type", "phase_score": "prob"}, inplace=True)
picks["id"] = picks["id"].astype(str)
print(f"\n{len(picks)} picks on the station channels: " + ", ".join(f"{k} {v}" for k, v in picks["type"].value_counts().items()))
print("Pick format:", picks.iloc[:5], sep="\n")

## Region. The example derives the box from the station extent, but a single fiber is a 20 km line
## and the Rainier seismicity sits 10-15 km north of it, so the box is given explicitly: the same
## lat 45-48, lon -123 to -120, 0-40 km box as the PyOcto `events_octo_das` run (~230 x 330 km).
x0 = stations["longitude"].median()
y0 = stations["latitude"].median()
config = {}
config["center"] = (x0, y0)
config["xlim_degree"] = (-123, -120)
config["ylim_degree"] = (45, 48)


## projection to km
proj = Proj(f"+proj=aeqd +lon_0={config['center'][0]} +lat_0={config['center'][1]} +units=km")
stations[["x(km)", "y(km)"]] = stations.apply(lambda x: pd.Series(proj(longitude=x.longitude, latitude=x.latitude)), axis=1)
stations["z(km)"] = stations["elevation_m"].apply(lambda x: -x / 1e3)

### setting GMMA configs
config["use_dbscan"] = True
config["use_amplitude"] = False           # ELEP picks carry no amplitude
config["method"] = "BGMM"
if config["method"] == "BGMM":  ## BayesianGaussianMixture
    config["oversample_factor"] = 5
if config["method"] == "GMM":  ## GaussianMixture
    config["oversample_factor"] = 1
#!!Prior for (time and amplitude): set a larger value to prevent splitting events. A too large value will cause merging events.
# config["covariance_prior"] = [5.0, 2.0]

# earthquake location
config["vel"] = {"p": 6.0, "s": 6.0 / 1.75}       # only used for DBSCAN scaling once eikonal is set
config["dims"] = ["x(km)", "y(km)", "z(km)"]
config["x(km)"] = proj(longitude=config["xlim_degree"], latitude=[config["center"][1]] * 2)[0]
config["y(km)"] = proj(longitude=[config["center"][0]] * 2, latitude=config["ylim_degree"])[1]
config["z(km)"] = (0, 40)
config["bfgs_bounds"] = (
    (config["x(km)"][0] - 1, config["x(km)"][1] + 1),  # x
    (config["y(km)"][0] - 1, config["y(km)"][1] + 1),  # y
    (0, config["z(km)"][1] + 1),  # z
    (None, None),  # t
)

# DBSCAN:
##!!Truncate the picks into segments: change the dbscan_eps to balance speed and event splitting.
## estimate_eps() scales with the nearest-station spacing, which for DAS channels 96 m apart gives a
## few hundredths of a second and would split every event, so eps is set by hand: 10 s spans the
## whole P-to-S moveout along the 20 km fiber for a local event.
print(f"\nestimate_eps would give {estimate_eps(stations, config['vel']['p']):.3f} s; using 10 s")
config["dbscan_eps"] = 10.0
config["dbscan_min_samples"] = 3

## using Eikonal for 1D velocity model: IASP91 down to the bottom of the search depth
velocity_model = pd.read_csv(iasp91_csv)                      # columns depth, r, vp, vs
velocity_model = velocity_model[velocity_model["depth"] <= config["z(km)"][1]].copy()
# IASP91 repeats the depth at layer boundaries; np.interp needs increasing depths, so nudge duplicates
velocity_model["depth"] += velocity_model.groupby("depth").cumcount() * 1e-3
vel = {"z": velocity_model["depth"].values, "p": velocity_model["vp"].values, "s": velocity_model["vs"].values}
h = 1.0
config["eikonal"] = {"vel": vel, "h": h, "xlim": config["x(km)"], "ylim": config["y(km)"], "zlim": config["z(km)"]}

# set number of cpus
config["ncpu"] = min(16, os.cpu_count())

##!!Post filtering (indepent of gmm): change these parameters to filter out associted picks with large errors
config["min_picks_per_eq"] = 100          # same as the PhaseNet-DAS reference run (gamma_association_11022026)
config["min_p_picks_per_eq"] = 0
config["min_s_picks_per_eq"] = 0
config["max_sigma11"] = 2.0  # second
config["max_sigma22"] = 1.0  # log10(m/s)
config["max_sigma12"] = 1.0  # covariance

for k, v in config.items():
    if k == "eikonal":
        print(f"{k}: IASP91 {len(vel['z'])} layers, h={h} km, zlim={v['zlim']}")
    else:
        print(f"{k}: {v}")

window 2023-08-27 00:00:00+00:00 .. 2023-08-28 00:00:00+00:00: 24 hourly files
320 PhaseNet-DAS station channels; 0 have no located ELEP neighbour and are dropped
320 DAS stations out of 3191 located channels
Station format:
   id  longitude  latitude  elevation_m
0   2 -121.73695  46.78581       1667.0
1  10 -121.73703  46.78563       1668.0
2  22 -121.73841  46.78538       1664.0
3  30 -121.73922  46.78498       1662.0
4  42 -121.73974  46.78407       1655.0
read and QC'd 24 pick files in 3 s
pick QC (edge_s=1 s, min_run=7, max_dt=0.5 s), all ELEP channels:
  112 of 1440 1-minute records kept (7.8%)
  435813 picks: 12512 dropped at a record boundary, 392411 dropped in records without a coherent run, 30890 kept (7.1%)
  11303 kept picks on the 320 station channels

11303 picks on the station channels: P 6321, S 4982
Pick format:
     id type                        timestamp  prob
0  1110    P 2023-08-27 00:59:02.073645+00:00   1.0
1  2982    P 2023-08-27 00:59:02.193645+00:00   1.0
2 

## 2. Association with GaMMA

In [5]:
import time
t0 = time.time()
event_idx0 = 0  ## current earthquake index
assignments = []
events, assignments = association(picks, stations, config, event_idx0, config["method"])
event_idx0 += len(events)
print(f"{len(events)} events from {len(assignments)} associated picks in {time.time() - t0:.0f} s")

## create catalog
events = pd.DataFrame(events)
if len(events):
    events[["longitude", "latitude"]] = events.apply(lambda x: pd.Series(proj(longitude=x["x(km)"], latitude=x["y(km)"], inverse=True)), axis=1)
    events["depth_km"] = events["z(km)"]
    events.to_csv(result_path("gamma_events.csv"), index=False,
                  float_format="%.3f",
                  date_format="%Y-%m-%dT%H:%M:%S.%f")

## add assignment to picks
assignments = pd.DataFrame(assignments, columns=["pick_index", "event_index", "gamma_score"])
picks = picks.join(assignments.set_index("pick_index")).fillna(-1).astype({"event_index": int})
picks.rename(columns={"id": "station_id", "timestamp": "phase_time", "type": "phase_type", "prob": "phase_score"}, inplace=True)
picks.to_csv(result_path("gamma_picks.csv"), index=False,
             date_format="%Y-%m-%dT%H:%M:%S.%f")

if len(events):
    n_by_phase = picks[picks["event_index"] >= 0].groupby(["event_index", "phase_type"]).size().unstack(fill_value=0)
    events = events.merge(n_by_phase.rename(columns={"P": "n_p", "S": "n_s"}), left_on="event_index", right_index=True, how="left")
    cols = ["event_index", "time", "longitude", "latitude", "depth_km", "num_picks", "n_p", "n_s", "sigma_time", "gamma_score"]
    events["time"] = pd.to_datetime(events["time"], utc=True)
    per_day = events.set_index("time").resample("1D").size()
    print(f"events per day: median {per_day.median():.0f}, max {per_day.max()} on {per_day.idxmax().date()}; "
          f"{(events['n_p'] == 0).sum()} S-only events, {(events['n_s'] == 0).sum()} P-only events")
    with pd.option_context("display.width", 200, "display.max_columns", 20):
        print(events[[c for c in cols if c in events.columns]].head(30).to_string(index=False))

Eikonal Solver: 
Iter 0, error = 999.828
Iter 1, error = 0.000
Time: 2.397
Eikonal Solver: 
Iter 0, error = 999.702
Iter 1, error = 0.000
Time: 0.018


Clustering (eps=8.33): 100%|██████████| 105/105 [00:00<00:00, 18627.16it/s]


Associating 104 clusters with 16 CPUs
........................................................................................................x
x
4 events from 1009 associated picks in 113 s
events per day: median 4, max 4 on 2023-08-27; 0 S-only events, 0 P-only events
 event_index                             time   longitude  latitude  depth_km  num_picks  n_p  n_s  sigma_time  gamma_score
           0 2023-08-27 05:36:03.543000+00:00 -121.954341 46.793671  7.020295        253   87  166    0.126982   246.712051
           1 2023-08-27 11:16:30.935000+00:00 -121.942677 46.789599  7.740584        190   58  132    0.139334   188.301513
           2 2023-08-27 10:10:21.838000+00:00 -121.949591 46.788267  7.585681        436  172  264    0.116032   419.417551
           3 2023-08-27 11:35:11.797000+00:00 -121.942282 46.793229  7.183343        130   37   93    0.145288   124.316532


In [6]:
import pandas as pd

In [7]:
gamma_events = pd.read_csv(result_path("gamma_events.csv"))
gamma_events

,time,magnitude,sigma_time,sigma_amp,cov_time_amp,gamma_score,num_picks,num_p_picks,num_s_picks,event_index,x(km),y(km),z(km),longitude,latitude,depth_km
0,2023-08-27T05:36:03.543,999,0.127,0,0,246.712,253,87,166,0,-12.478,2.677,7.020,-121.954,46.794,7.020
1,2023-08-27T11:16:30.935,999,0.139,0,0,188.302,190,58,132,1,-11.588,2.222,7.741,-121.943,46.790,7.741
2,2023-08-27T10:10:21.838,999,0.116,0,0,419.418,436,172,264,2,-12.116,2.075,7.586,-121.950,46.788,7.586
3,2023-08-27T11:35:11.797,999,0.145,0,0,124.317,130,37,93,3,-11.557,2.626,7.183,-121.942,46.793,7.183
